In [0]:
# Databricks Notebook — 02_silver_transform.py
# Layer   : Silver
# Purpose : Schema validation & casting, null/outlier handling,
#           deduplication, join identity + transactions → write Delta Lake.
# Run order: 2 of 3  (requires 01_bronze_reader to have run)

from pyspark.sql import functions as F
from pyspark.sql.types import (
    DoubleType, IntegerType, LongType
)

# ---------------------------------------------------------------------------
# 0. CONFIGURATION  ← must match 01_bronze_reader.py exactly
# ---------------------------------------------------------------------------
STORAGE_ACCOUNT = os.getenv("STORAGE_ACCOUNT")
STORAGE_KEY     = os.getenv("STORAGE_KEY")

CONTAINER    = "raw-data"
SILVER_BASE  = f"abfss://{CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net/fraud_silver"

IDENTITY_DELTA_PATH     = f"{SILVER_BASE}/delta/identity/"
TRANSACTIONS_DELTA_PATH = f"{SILVER_BASE}/delta/transactions/"
JOINED_DELTA_PATH       = f"{SILVER_BASE}/delta/joined/"

DATABASE = "fraud_lakehouse"   # must match Bronze

# ---------------------------------------------------------------------------
# 1. RE-APPLY ADLS Gen2 AUTHENTICATION + DELTA OPTIMIZATIONS
# ---------------------------------------------------------------------------
spark.conf.set(
    f"fs.azure.account.key.{STORAGE_ACCOUNT}.dfs.core.windows.net",
    STORAGE_KEY
)
spark.conf.set("spark.databricks.delta.optimizeWrite.enabled", "true")
spark.conf.set("spark.databricks.delta.autoCompact.enabled",   "true")
spark.conf.set("spark.sql.adaptive.enabled",                   "true")

spark.sql(f"USE {DATABASE}")

print("✅  ADLS Gen2 authentication configured.")

In [0]:
# ---------------------------------------------------------------------------
# 2. READ BRONZE DELTA TABLES FROM METASTORE
#    spark.table() reads from the Delta metastore — no paths, no temp views,
#    no session state issues. Both DFs are cached for all downstream actions.
# ---------------------------------------------------------------------------
identity_raw     = spark.table(f"{DATABASE}.bronze_identity").cache()
transactions_raw = spark.table(f"{DATABASE}.bronze_transactions").cache()

# Guard: abort early if Bronze tables are empty
assert identity_raw.count() > 0,     "Silver abort: bronze_identity Delta table is empty!"
assert transactions_raw.count() > 0, "Silver abort: bronze_transactions Delta table is empty!"

print(f"✅  bronze_identity     : {identity_raw.count():,} rows loaded from Delta.")
print(f"✅  bronze_transactions : {transactions_raw.count():,} rows loaded from Delta.")

In [0]:
# ---------------------------------------------------------------------------
# 3. SCHEMA VALIDATION & CASTING
# ---------------------------------------------------------------------------

# ── 3a. Identity ─────────────────────────────────────────────────────────────
#   Columns: TransactionID, id_01..id_38, DeviceType, DeviceInfo
identity_df = (
    identity_raw
    .withColumn("TransactionID", F.col("TransactionID").cast(LongType()))
    .withColumn("DeviceType",    F.trim(F.upper(F.col("DeviceType"))))
    .withColumn("DeviceInfo",    F.trim(F.col("DeviceInfo")))
)

print("✅  Identity schema cast complete.")

# ── 3b. Transactions ─────────────────────────────────────────────────────────
#   Columns: TransactionID, isFraud, TransactionDT, TransactionAmt,
#            ProductCD, card1..card6, addr1, addr2, dist1, dist2,
#            P_emaildomain, R_emaildomain, C1..C14, D1..D15, M1..M9, V1..V339
transactions_df = (
    transactions_raw
    .withColumn("TransactionID",  F.col("TransactionID").cast(LongType()))
    .withColumn("isFraud",        F.col("isFraud").cast(IntegerType()))
    .withColumn("TransactionDT",  F.col("TransactionDT").cast(LongType()))
    .withColumn("TransactionAmt", F.col("TransactionAmt").cast(DoubleType()))
    .withColumn("ProductCD",      F.trim(F.col("ProductCD")))
    .withColumn("card1",          F.col("card1").cast(IntegerType()))
    .withColumn("card2",          F.col("card2").cast(DoubleType()))
    .withColumn("addr1",          F.col("addr1").cast(DoubleType()))
    .withColumn("addr2",          F.col("addr2").cast(DoubleType()))
    .withColumn("dist1",          F.col("dist1").cast(DoubleType()))
    .withColumn("dist2",          F.col("dist2").cast(DoubleType()))
    # Derived readable timestamp (START_DATE = 2017-11-30 for IEEE dataset)
    .withColumn(
        "TransactionTimestamp",
        (F.to_timestamp(F.lit("2017-11-30")) + F.expr("INTERVAL 1 second") * F.col("TransactionDT"))
    )
)

print("✅  Transactions schema cast complete.")

In [0]:
# ---------------------------------------------------------------------------
# 4. NULL / OUTLIER HANDLING
# ---------------------------------------------------------------------------

# ── 4a. Identity — drop rows without a join key
identity_clean = identity_df.dropna(subset=["TransactionID"])

# ── 4b. Transactions — critical fields must not be null
transactions_clean = (
    transactions_df
    .dropna(subset=["TransactionID", "TransactionAmt", "isFraud"])
    .filter(F.col("TransactionAmt") > 0)
    # Cap extreme outliers at 99th percentile (optional — uncomment if needed)
    # .filter(F.col("TransactionAmt") < transactions_df.approxQuantile("TransactionAmt", [0.99], 0.01)[0])
)

dropped_txn = transactions_raw.count() - transactions_clean.count()
print(f"✅  Null/outlier removal: dropped {dropped_txn:,} transaction rows.")


In [0]:
# Rows where ALL of these are null at the same time
transactions_clean_v1 = transactions_clean.filter(F.col("card4").isNull())
display(transactions_clean_v1)

In [0]:
# ---------------------------------------------------------------------------
# 5. DEDUPLICATION
# ---------------------------------------------------------------------------
identity_deduped     = identity_clean.dropDuplicates(["TransactionID"])
transactions_deduped = transactions_clean.dropDuplicates(["TransactionID"])

dup_id  = identity_clean.count()     - identity_deduped.count()
dup_txn = transactions_clean.count() - transactions_deduped.count()
print(f"✅  Deduplication: removed {dup_id:,} identity dupes, {dup_txn:,} transaction dupes.")


In [0]:

# ---------------------------------------------------------------------------
# 6. JOIN IDENTITY + TRANSACTIONS  (left join — keep all transactions)
# ---------------------------------------------------------------------------
joined_df = (
    transactions_deduped
    .join(identity_deduped, on="TransactionID", how="left")
)

print(f"✅  Joined Silver dataset rows: {joined_df.count():,}")

In [0]:
# ---------------------------------------------------------------------------
# 6. JOIN IDENTITY + TRANSACTIONS  (left join — keep all transactions)
# ---------------------------------------------------------------------------
joined_df = (
    transactions_deduped
    .join(identity_deduped, on="TransactionID", how="left")
)

print(f"✅  Joined Silver dataset rows: {joined_df.count():,}")

In [0]:
# ---------------------------------------------------------------------------
# 7. WRITE SILVER DELTA TABLES
#    Each table is partitioned by ProductCD for partition pruning in Gold.
#    saveAsTable registers in the metastore — Gold reads with spark.table()
#    and no path or session state dependency.
# ---------------------------------------------------------------------------

# ── Identity
(
    identity_deduped
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .option("path", IDENTITY_DELTA_PATH)
    .saveAsTable(f"{DATABASE}.silver_identity")
)

# ── Transactions (partitioned — common filter in Gold aggregations)
(
    transactions_deduped
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .partitionBy("ProductCD")
    .option("path", TRANSACTIONS_DELTA_PATH)
    .saveAsTable(f"{DATABASE}.silver_transactions")
)

# ── Joined (primary Silver output for Gold — partitioned by ProductCD)
(
    joined_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .partitionBy("ProductCD")
    .option("path", JOINED_DELTA_PATH)
    .saveAsTable(f"{DATABASE}.silver_joined")
)


In [0]:
# ---------------------------------------------------------------------------
# 8. OPTIMIZE DELTA TABLES  (compact small files for fast Gold reads)
# ---------------------------------------------------------------------------
spark.sql(f"OPTIMIZE {DATABASE}.silver_identity")
spark.sql(f"OPTIMIZE {DATABASE}.silver_transactions ZORDER BY (TransactionID)")
spark.sql(f"OPTIMIZE {DATABASE}.silver_joined       ZORDER BY (TransactionID, card1)")

print(f"\n✅  Silver Delta tables written to : {SILVER_BASE}/delta/")
print(f"    ├── silver_identity     : {spark.table(f'{DATABASE}.silver_identity').count():,} rows")
print(f"    ├── silver_transactions : {spark.table(f'{DATABASE}.silver_transactions').count():,} rows")
print(f"    └── silver_joined       : {spark.table(f'{DATABASE}.silver_joined').count():,} rows")
print(f"\n✅  All tables registered in metastore database '{DATABASE}'")
print("▶   Run notebook 03_gold_aggregations.py next.")